# Pré processamento dos dados

## Introdução

Este notebook apresenta o passo a passo para a obtenção dos dados brutos das fontes de dados utilizadas no MVP, assim como o seu pré-processamento para permitir a ingestão deles em tabelas do databricks. Todas os processos dessa etapa foram realizadas fora do databricks.

As planilhas baixadas estão salvas no volume raw.mvp_data.

Os arquivos csv gerados estão salvos no volume raw.mvp_raw.

As ingestões nos volumes foram realizadas a partir da ferramenta "Data Ingestion".

![](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/ingestao_planilhas.png)

![](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/ingestao_raw.png)

## Dados do Sistema de Informação de Mortalidade (SIM)

### 1. Coleta de Dados do SIM

Os dados do **SIM** foram obtidos através do R com a biblioteca `microdatasus`. Essa biblioteca "baixa, lê e prepara microdados dos Sistemas de Informação em Saúde publicados pelo DataSUS. O pacote trabalha diretamente com arquivos DBC, seleciona os arquivos efetivamente disponíveis no servidor e oferece funções específicas para recodificar variáveis de SIM, SINASC, SIH, SIA, CNES e SINAN."

**Documentação oficial:** [github.com/rfsaldanha/microdatasus](https://github.com/rfsaldanha/microdatasus?utm_source=gemini)

#### 2. Código R

```
# Instalação do pacote
install.packages("microdatasus")

# Carregamento da biblioteca
library(microdatasus)

# Coleta dos dados do SIM para a Bahia (2020-2025)
sim_raw <- fetch_datasus(
  year_start = 2020,
  year_end = 2025,
  uf = "BA",
  information_system = "SIM-DO"
)

# Processamento e recodificação das variáveis
sim <- process_sim(sim_raw)

# Salvar os dados na camada raw
write.csv(sim, "raw/sim_raw.csv", row.names = FALSE)
```

## Dados do IBGE

Os dados do IBGE referentes às informações dos municípios foram extraídos diretamente do site oficial.

### 1. Acesso e Download

Os dados podem ser acessados através do link: [Códigos dos Municípios - IBGE](https://www.ibge.gov.br/explica/codigos-dos-municipios.php?utm_source=gemini).

Na página **"Códigos dos municípios IBGE"**, foi selecionada a opção **"Download da tabela de códigos"**.

![site_ibge.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/site_ibge.png)

### 2. Extração dos Arquivos

Ao selecionar essa opção, é efetuado o download de um arquivo `.zip` contendo a seguinte estrutura:

![zip_ibge.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/zip_ibge.png)

A partir dele, acessa-se a pasta `DTB_2023` para utilizar o arquivo em formato Excel denominado `RELATORIO_DTB_BRASIL_MUNICIPIO.xls`.

![arquivos_ibge.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/arquivos_ibge.png)

### 3. Estrutura e Pré-processamento

O arquivo apresenta as características ilustradas na imagem abaixo. Nota-se a presença de um cabeçalho inicial que precisará ser removido durante a etapa de leitura dos dados, garantindo que a estrutura fique em um formato compatível para a ingestão de tabelas no **Databricks**.

![planilha_ibge.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/planilha_ibge.png)

### 4. Codigo

```
import pandas as pd

# ler o arquivo removendo as 6 primeiras linhas de cabeçalho
ibge = pd.read_excel("raw/RELATORIO_DTB_BRASIL_MUNICIPIO.xls", skiprows=6)
display(ibge)

# Salvar os dados na camada raw
ibge.to_csv("raw/ibge_raw.csv", index=False)

```

## Dados do CID

Os dados de concessões do CID-10 foram obtidos a partir do portal do Ministério da Previdência Social.

### 1. Acesso e Download

Os dados podem ser acessados através do link: [Tabelas CID-10 - Ministério da Previdência Social](https://www.gov.br/previdencia/pt-br/assuntos/previdencia-social/saude-e-seguranca-do-trabalhador/acidente_trabalho_incapacidade/tabelas-cid-10?utm_source=gemini).

Na seção referente ao ano de **2025**, realiza-se o download direto da planilha na opção **"Previdenciários, janeiro a dezembro"**, obtendo o arquivo em formato Excel denominado `quantidade-de-b31-por-cid-concessao-2025-excel.xlsx`.

![site_cid.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/site_cid.png)

### 2. Estrutura e Pré-processamento

O arquivo baixado possui a estrutura exemplificada abaixo. Assim como na fonte do IBGE, é realizada a remoção das primeiras linhas (cabeçalho inicial) durante a leitura do arquivo para adequar a tabela ao formato aceito no **Databricks**.

![planilha_cid.png](/Volumes/mvp_engenharia_dados/raw/mvp_imgs/planilha_cid.png)

### 3. Código

```
import pandas as pd

# Ler arquivo e remover as primeiras linhas de cabeçalho
cid = pd.read_excel("raw/quantidade-de-b31-por-cid-concessao-2025-excel.xlsx", skiprows=12)
display(cid)

# Renomear colunas para formatos permitido no delta table do databricks
dic_colunas = {
    'Unnamed: 0' : 'Coluna 1',
    'Unnamed: 1' : 'Coluna 2'
}

cid = cid.rename(columns=dic_colunas)

# Salvar os dados na camada raw
cid.to_csv("raw/cid_raw.csv", index=False)
```

